# s21 — Skills

**What this teaches:** how to give an agent a library of *playbooks* it can pull on demand. A skill is a markdown file with a tiny YAML front matter; the `skills` toolset turns the directory into two tools — `list_skills` (what's available) and `load_skill` (read one into context). The model decides when to load, so you can ship deep domain knowledge without burning every token on it upfront.

**Concept anchor:** skills make the agent retargetable. The same binary becomes a code reviewer, a Kubernetes triager, or a DBA helper depending on which `skills/` directory you mount.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars before launching Jupyter, e.g.:
  ```
  export OMNIS_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export OMNIS_MODEL=qwen2.5-coder
  ```
  Defaults to a local Ollama. Swap in `anthropic` / `openai` / `gemini` for hosted providers — see [docs/providers.md](../../docs/providers.md).
- A `skills/` directory in the working directory containing one or more `SKILL.md` files (the repo ships some examples).

## 1. Imports

We pull in `internal/skills` for the toolset constructor and ADK's `tool.Toolset` interface so we can hand the toolset to the agent.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "google.golang.org/adk/tool"

    "github.com/blouargant/omnis/core/agentkit"
    "github.com/blouargant/omnis/core/stream"
    "github.com/blouargant/omnis/internal/skills"
)

## 2. Helper

We swap the example's `os.Exit`-based `must` for a panic version so a notebook error doesn't kill the GoNB kernel.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Build the skills toolset

`skills.Toolset` scans the given directory for `<name>/SKILL.md` files. Each file looks like:

```
---
name: review-go
description: Checklist for reviewing Go code.
---
# Body of the playbook in markdown...
```

Only the YAML front matter (`name`, `description`) is shown by `list_skills` — the markdown body is fetched lazily by `load_skill`, so a directory of fifty skills costs you near-zero context until the model actually opens one.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
ts, err := skills.Toolset(ctx, "skills")
must(err)
fmt.Println("skills toolset wired from ./skills/")

## 4. Wire the agent

Toolsets are attached via `Toolsets` (not `Tools`) — they're allowed to add or remove tools dynamically. The instruction nudges the model to check `list_skills` before answering, otherwise it will often try to answer from memory and skip the playbook entirely.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s21_skills",
    Description: "Skill-aware agent.",
    Model:       llm,
    Toolsets:    []tool.Toolset{ts},
    Instruction: "When the user asks about something a known skill covers, call load_skill first.",
})
must(err)
r, err := agentkit.Runner("s21", a)
must(err)

## 5. Run a prompt that needs a skill

Ask for something a skill is likely to cover. Watch the stream: you should see `list_skills` called once, then `load_skill` for the chosen entry, then the model's answer grounded in the playbook.

In [ ]:
prompt := "List your available skills and pick the most relevant for reviewing Go code."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- The very first tool call is `list_skills` — the model can't load what it can't see. If you see `load_skill` straight away, the model has guessed at a name; tighten the instruction.
- Skills and soft-skills are *peers*: [s23_softskills](../s23_softskills/s23_softskills.ipynb) shows the curator producing similar markdown automatically from past sessions.
- Compare with [s10_mcp](../s10_mcp/s10_mcp.ipynb) for the *other* way to bolt domain capabilities onto the same loop — MCP servers expose tools, skills expose prose.

## Try it yourself

1. Drop a new `skills/dba/SKILL.md` with a short PostgreSQL triage checklist and ask the agent a Postgres question — it should auto-discover the new file on next run.
2. Remove the `Instruction` line and re-run the prompt. Many models will answer without consulting `list_skills` at all — proof that the instruction, not the toolset, is what makes skills *actually* used.